# Week 10 - Time Series Data

**Paññāsāstra University of Cambodia**
**Course:** Data Analysis and Visualization (3 credits)
**Chapter 10 — Time Series Data**

---

## Session topics (from the course outline)

- Working with dates and times
- Resampling and frequency analysis
- Plotting time series data

## Course learning outcomes addressed

| CLO | Statement |
|---|---|
| **CLO 3** | Apply Python libraries (NumPy, Pandas) to clean, manipulate and process datasets |
| **CLO 6** | Generate statistical summaries to describe data patterns |
| **CLO 8** | Evaluate the effectiveness of different visualization methods for communicating insights |

## Reading

| Topic | Source |
|---|---|
| Date and time data types and tools | McKinney §11.1 |
| Time series basics; indexing and selection | McKinney §11.2 |
| Date ranges, frequencies and shifting | McKinney §11.3 |
| Resampling and frequency conversion | McKinney §11.6 |
| Moving window functions | McKinney §11.7 |
| Working with dates and times | Pfaffelhuber §7.10 |

## What is different about time

Every dataset so far had rows that could be shuffled without losing anything. A time series
cannot: the order *is* information, and so is the spacing between observations.

That gives pandas something to work with. Once the index is a `DatetimeIndex`, you can select
a month by naming it, change the sampling frequency in one call, and compute rolling windows
without writing a loop - the loop you did write, by hand, in Week 3.

The session's central point is a familiar one arriving in a new form. **Choosing a frequency
is choosing what to discard**, and the choice can hide the most important pattern in your
data entirely.

---

## Setup

All imports for this notebook are defined in the single cell below. Run it first. If any
import fails, return to Notebook 00 and check that the kernel in the top-right corner points
inside your `venv`.

In [ ]:
# ---------------------------------------------------------------------------
# All imports for this notebook. Run this cell first.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility: every dataset in this notebook is generated from this seed,
# so your numbers will match the ones shown in the lecture exactly.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Consistent, colour-blind-safe plotting defaults for the whole course.
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 100,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})
sns.set_palette(PALETTE)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Environment ready.")
print(f"  numpy   {np.__version__}")
print(f"  pandas  {pd.__version__}")
print(f"  seaborn {sns.__version__}")
print(f"  seed    {RANDOM_SEED}")

---

# Part 1 — Theory and concepts

## 1.1 The three time types

| Type | Represents | Built by |
|---|---|---|
| `Timestamp` | A moment | `pd.Timestamp("2026-03-02")` |
| `Timedelta` | A duration | `pd.Timedelta(days=7)`, or subtracting two timestamps |
| `Period` | A span with a frequency | `pd.Period("2026-03", freq="M")` |

`Timestamp` is what you will use almost always. It is pandas' version of Python's
`datetime`, and it prints the same way.

Arithmetic behaves as you would expect: a timestamp minus a timestamp is a `Timedelta`, and a
timestamp plus a `Timedelta` is another timestamp.

## 1.2 Parsing

```python
pd.to_datetime(series)                                   # infer the format
pd.to_datetime(series, format="%d/%m/%Y")                # state it
pd.to_datetime(series, errors="coerce")                  # unparseable -> NaT
pd.read_csv(path, parse_dates=["date"])                  # at read time
```

Two habits worth forming.

**State the format when you know it.** Inference is slow on a large column, and it is
ambiguous: `03/04/2026` is 3 April in most of the world and 4 March in the United States.
pandas guesses from the whole column, so a file that happens to contain `13/04/2026` is read
one way and a file that does not is read the other. Passing `format=` removes the guess.

**Use `errors="coerce"` on data you did not create.** Anything unparseable becomes `NaT`
rather than raising, and counting the `NaT` values tells you how bad the column is.

The missing marker for datetimes is `NaT`, not `NaN`. `pd.isna()` recognises both;
`np.isnan()` raises on `NaT`.

## 1.3 The `.dt` accessor

For a column of timestamps, `.dt` exposes the parts:

```python
df["date"].dt.year      .dt.month      .dt.day
df["date"].dt.dayofweek       # Monday = 0 ... Sunday = 6
df["date"].dt.day_name()      # "Monday"
df["date"].dt.quarter         .dt.is_month_end
```

`.dt` is to datetimes what `.str` is to strings: the accessor that applies an element-wise
operation to the whole column.

## 1.4 The `DatetimeIndex`

Putting the dates in the index unlocks most of what makes pandas good at time series:

```python
df = df.set_index("date").sort_index()
```

**Sort it.** Nearly every time-series operation assumes the index is ordered, and several give
wrong answers rather than errors when it is not.

**Partial string indexing** then works — you select a period by naming it:

```python
df.loc["2026"]              # a whole year
df.loc["2026-03"]           # a whole month
df.loc["2026-03-02"]        # one day
df.loc["2026-03":"2026-04"] # a range, INCLUSIVE of both ends
```

This is the `.loc` label-slicing rule from Week 4: label slices include both endpoints.

## 1.5 Resampling

Resampling changes the frequency of the series. It is `groupby` for time.

```python
daily.resample("W").mean()        # downsample: many rows -> fewer
hourly.resample("D").sum()
monthly.resample("D").ffill()     # upsample: few rows -> more
```

**Downsampling** combines several observations into one and needs an aggregation: `mean`,
`sum`, `max`, `count`, or `.agg([...])` for several at once. Which one is correct depends on
the quantity — a **sum** for counts and revenue, a **mean** for rates and temperatures. Taking
the mean of a daily sales total gives you the average day, not the week's takings.

**Upsampling** creates rows that have no data, and you must say what fills them: `ffill`,
`bfill`, `interpolate`, or leave them as `NaN`. Upsampling invents nothing on its own, and
`ffill` is a claim that the value did not change.

### Frequency aliases, and a change that breaks old code

| Alias | Meaning |
|---|---|
| `D` | Calendar day |
| `W` | Week, ending Sunday |
| `ME` | Month **end** |
| `MS` | Month **start** |
| `QE` | Quarter end |
| `YE` | Year end |
| `h` | Hour |
| `min` | Minute |
| `7D`, `2W` | Any multiple |

**pandas 2.2 renamed several of these, and pandas 3.0 removed the old names.**

| Old | New | In pandas 3.0 |
|---|---|---|
| `M` | `ME` | raises `ValueError` |
| `Q` | `QE` | raises `ValueError` |
| `Y` or `A` | `YE` | raises `ValueError` |
| `H` | `h` | raises `ValueError` |

Nearly every tutorial and answer you find online still uses `M` and `H`. If you copy one and
get `ValueError: Invalid frequency: M`, this is why, and the fix is one letter.

## 1.6 Rolling windows

```python
series.rolling(7).mean()                  # 7-observation window
series.rolling(7, center=True).mean()      # centred rather than trailing
series.rolling("7D").mean()                # 7 calendar days, gaps allowed
series.expanding().mean()                  # everything up to each point
```

This is the `moving_average` function you wrote by hand in Week 3, as one method call.

Two details:

- **The first `window - 1` values are `NaN`**, because a full window does not yet exist. That
  is correct behaviour, not a defect: a 7-day average of 3 days would be a different quantity.
  `min_periods=` relaxes it if you are willing to accept partial windows.
- **`center=True` shifts the window** so the result lines up with the middle of the period
  rather than its end. Use it for describing the past; leave it off for anything that must not
  peek at the future, such as a forecast.

## 1.7 Comparing with the past

```python
series.shift(1)             # yesterday's value, on today's row
series.diff()               # today minus yesterday
series.pct_change()         # proportional change
series.diff(7)              # change against the same day last week
```

`shift` is the building block: `diff` is `series - series.shift(1)`.

On seasonal data, comparing with the immediately preceding period mixes the seasonal cycle
into the result. If Mondays are always busy, `diff()` on daily data mostly measures the day of
the week. `diff(7)` compares like with like.

---

---

# Part 2 — Code example

§2.5 is the centre of the session.

## 2.1 A series with structure at three scales

In [ ]:
def generate_library_visits(seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Build one year of daily library visits with three layers of structure.

    The layers are deliberately at different time scales, so that resampling
    to different frequencies reveals some and destroys others:

      - a slow upward TREND across the year          (visible at any frequency)
      - a strong WEEKLY cycle, quiet at weekends     (destroyed by monthly resampling)
      - two EXAMINATION PEAKS a few weeks long       (visible weekly, blurred monthly)

    Returns
    -------
    pandas.DataFrame
        Columns: date, visits, is_weekend. One row per day.
    """
    generator = np.random.default_rng(seed)

    dates = pd.date_range("2026-01-01", "2026-12-31", freq="D")
    day_index = np.arange(len(dates))
    weekday = dates.dayofweek.to_numpy()

    trend = 260 + day_index * 0.42
    weekly = np.where(weekday >= 5, -130, 40)

    exams = np.zeros(len(dates))
    for peak_day, height in [(150, 180), (330, 220)]:
        exams += height * np.exp(-((day_index - peak_day) ** 2) / 260)

    noise = generator.normal(0, 22, len(dates))
    visits = np.clip(trend + weekly + exams + noise, 0, None).round().astype(int)

    return pd.DataFrame({
        "date": dates,
        "visits": visits,
        "is_weekend": weekday >= 5,
    })


raw = generate_library_visits()

print(f"{len(raw)} daily observations, "
      f"{raw['date'].min():%d %b %Y} to {raw['date'].max():%d %b %Y}")
print(raw.head())
print(f"\nColumn dtype: {raw['date'].dtype}")

## 2.2 Parsing, and the ambiguity that silently changes your data

In [ ]:
# Dates usually arrive as text.
as_text = pd.Series(["2026-03-02", "2026-03-03", "not a date", "2026-03-05"])

print("Default parsing raises on the bad value:")
try:
    pd.to_datetime(as_text)
except (ValueError, pd.errors.ParserError) as exc:
    print(f"  {type(exc).__name__}: {str(exc)[:70]}...")

coerced = pd.to_datetime(as_text, errors="coerce")
print("\nWith errors='coerce':")
print(coerced)
print(f"\nunparseable values: {int(coerced.isna().sum())}")
print("\nThe missing marker for a datetime is NaT, not NaN. pd.isna() catches")
print("both; np.isnan() raises on NaT.")

In [ ]:
# The ambiguity that matters: is 03/04/2026 the 3rd of April or the 4th of March?
ambiguous = ["03/04/2026", "05/04/2026", "07/04/2026"]

day_first = pd.to_datetime(ambiguous, format="%d/%m/%Y")
month_first = pd.to_datetime(ambiguous, format="%m/%d/%Y")

print("The same three strings, parsed two ways:")
print(f"{'input':<14}{'day first':<16}{'month first'}")
print("-" * 46)
for text, d, m in zip(ambiguous, day_first, month_first):
    print(f"{text:<14}{d:%d %b %Y}      {m:%d %b %Y}")

print("\nBoth are valid readings. Neither raises. The dates differ by a month,")
print("and every downstream calculation differs with them.")
print("\npandas guesses from the whole column, so a file that happens to")
print("contain a day above 12 is read one way and a file that does not is")
print("read the other. STATE THE FORMAT when you know it.")

## 2.3 The `.dt` accessor and the `DatetimeIndex`

In [ ]:
# .dt is to datetimes what .str is to strings.
features = pd.DataFrame({
    "date": raw["date"],
    "year": raw["date"].dt.year,
    "month": raw["date"].dt.month,
    "day_name": raw["date"].dt.day_name(),
    "dayofweek": raw["date"].dt.dayofweek,
    "quarter": raw["date"].dt.quarter,
    "is_month_end": raw["date"].dt.is_month_end,
})
print(features.head(8).to_string(index=False))
print("\ndayofweek: Monday = 0 ... Sunday = 6")

In [ ]:
# Putting the dates in the index unlocks the time-series machinery.
visits = raw.set_index("date").sort_index()

print("Index type:", type(visits.index).__name__)
print("Sorted    :", visits.index.is_monotonic_increasing)

print("\nPartial string indexing — name a period and get it:")
print(f"  visits.loc['2026-03']       -> {len(visits.loc['2026-03']):>3} rows (March)")
print(f"  visits.loc['2026-03-02']    -> {len(visits.loc[['2026-03-02']]):>3} row")
print(f"  visits.loc['2026-03':'2026-04'] -> "
      f"{len(visits.loc['2026-03':'2026-04']):>3} rows (March AND April)")

print("\nThe range form includes BOTH endpoints — the .loc label-slicing rule")
print("from Week 4. There is no 'one past the end' label to name.")

print("\nSorting matters. Most time-series operations assume an ordered index,")
print("and several give a wrong answer rather than an error when it is not.")

## 2.4 Frequency aliases, and the rename that breaks old code

In [ ]:
series = visits["visits"]

print(f"pandas {pd.__version__}\n")
print("Frequency aliases that were renamed in pandas 2.2:")
print(f"{'old':<8}{'new':<8}{'status here'}")
print("-" * 46)

for old, new in [("M", "ME"), ("Q", "QE"), ("Y", "YE"), ("H", "h")]:
    try:
        series.resample(old).mean()
        status = "still accepted (older pandas)"
    except ValueError:
        status = "raises ValueError"
    print(f"{old:<8}{new:<8}{status}")

print("\nAll the new names work:")
for alias, label in [("D", "day"), ("W", "week"), ("ME", "month end"),
                     ("QE", "quarter end"), ("YE", "year end")]:
    print(f"  {alias:<5} {len(series.resample(alias).mean()):>4} periods   ({label})")

print("\nNearly every tutorial online still uses 'M' and 'H'. If you copy one")
print("and get 'ValueError: Invalid frequency: M', this is why. The fix is")
print("one letter.")

## 2.5 Resampling, and what each frequency destroys

The centre of the session.

In [ ]:
# The same series at four frequencies.
frequencies = {
    "daily": series,
    "weekly": series.resample("W").mean(),
    "monthly": series.resample("ME").mean(),
    "quarterly": series.resample("QE").mean(),
}

print(f"{'frequency':<12}{'observations':>14}{'mean':>9}{'std':>9}")
print("-" * 44)
for name, resampled in frequencies.items():
    print(f"{name:<12}{len(resampled):>14}{resampled.mean():>9.1f}{resampled.std():>9.1f}")

print("\nThe mean is almost unchanged at every frequency — averaging averages")
print("preserves the centre.")
print("\nThe standard deviation collapses. That is the variation being averaged")
print("away, and the next figure shows what was in it.")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)

for ax, (name, resampled) in zip(axes, frequencies.items()):
    ax.plot(resampled.index, resampled.to_numpy(), color=PALETTE[0], linewidth=1.2)
    ax.set_title(f"{name}  —  {len(resampled)} observations", loc="left", fontsize=10)
    ax.set_ylabel("visits")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("date")
fig.suptitle("One year of library visits, at four frequencies",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Daily    : a dense band. The weekly cycle is in there but unreadable.")
print("Weekly   : the two examination peaks appear clearly, and the trend.")
print("Monthly  : the trend survives; the peaks are blunted.")
print("Quarterly: four points. Only the trend is left.")

In [ ]:
# What exactly was destroyed: the weekly cycle.
by_day = raw.groupby(raw["date"].dt.day_name())["visits"].mean()
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]
by_day = by_day.reindex(day_order)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colours = [PALETTE[0]] * 5 + [PALETTE[3]] * 2
axes[0].bar(range(7), by_day.to_numpy(), color=colours, edgecolor="white")
axes[0].set_xticks(range(7), [d[:3] for d in day_order])
axes[0].set_ylabel("mean visits")
axes[0].set_title("(a) The weekly cycle, in the daily data", loc="left")
axes[0].grid(axis="x", alpha=0)

# One month, daily against the monthly average.
march = series.loc["2026-03"]
march_mean = march.mean()
axes[1].plot(march.index, march.to_numpy(), color=PALETTE[0],
             linewidth=1.4, marker="o", markersize=3, label="daily")
axes[1].axhline(march_mean, color=PALETTE[3], linewidth=2,
                linestyle="--", label=f"monthly mean {march_mean:.0f}")
axes[1].set_title("(b) March: 31 numbers become 1", loc="left")
axes[1].set_ylabel("visits")
axes[1].legend(fontsize=8, frameon=False)
axes[1].tick_params(axis="x", rotation=30, labelsize=8)

plt.tight_layout()
plt.show()

weekday_mean = raw.loc[~raw["is_weekend"], "visits"].mean()
weekend_mean = raw.loc[raw["is_weekend"], "visits"].mean()
print(f"Weekday mean: {weekday_mean:.0f}    Weekend mean: {weekend_mean:.0f}")
print(f"Weekends are {(1 - weekend_mean / weekday_mean) * 100:.0f}% quieter.")
print("\nThat is the single largest pattern in this dataset, and it is")
print("completely absent from the monthly series. Not reduced — absent.")
print("Every month contains whole weeks, so the weekend effect averages out")
print("exactly, leaving no trace.")
print("\nA reader given only the monthly chart cannot recover it, and has no")
print("way to know it is missing.")

In [ ]:
print("=" * 70)
print("CHOOSING A FREQUENCY IS CHOOSING WHAT TO DISCARD")
print("=" * 70)
print()
print("  A pattern is visible only at a frequency FINER than the pattern.")
print()
print(f"  {'pattern':<22}{'period':<14}{'visible at'}")
print("  " + "-" * 56)
print(f"  {'weekly cycle':<22}{'7 days':<14}daily")
print(f"  {'examination peaks':<22}{'~3 weeks':<14}daily, weekly")
print(f"  {'annual trend':<22}{'1 year':<14}daily, weekly, monthly, quarterly")
print()
print("  Resampling to monthly did not weaken the weekly cycle. It removed it,")
print("  because each month contains a whole number of weeks and the effect")
print("  cancels exactly.")
print()
print("  This is the Week 1 lesson again, and the Week 8 lesson again:")
print("  a summary discards information, and the question is always WHICH.")
print()
print("  Practical rule: look at the finest frequency you have FIRST, then")
print("  resample for presentation. Never the other way round.")

## 2.6 Choosing the aggregation

Downsampling needs a function, and the right one depends on what the numbers mean.

In [ ]:
weekly_mean = series.resample("W").mean()
weekly_sum = series.resample("W").sum()

print("The same week, two aggregations:")
comparison = pd.DataFrame({
    "mean": weekly_mean.head(4).round(1),
    "sum": weekly_sum.head(4),
})
print(comparison.to_string())

print("\nThe mean is 'visits on an average day that week'.")
print("The sum is 'visits that week'.")
print("\nBoth are correct and they answer different questions. For a count,")
print("the sum is usually what a reader assumes a weekly figure means.")

# Several aggregations at once.
weekly_summary = series.resample("W").agg(["count", "mean", "min", "max", "sum"])
print("\nSeveral at once with .agg:")
print(weekly_summary.head(4).round(1).to_string())

print("\nNote the count column: the first and last weeks are partial, because")
print("the year does not begin or end on a week boundary. Check it before")
print("comparing the first or last period with the others.")

## 2.7 Rolling windows

Week 3's `moving_average`, as one method call.

In [ ]:
rolling_7 = series.rolling(7).mean()
rolling_7_centred = series.rolling(7, center=True).mean()
rolling_30 = series.rolling(30, center=True).mean()

print("The first values are NaN, because a full window does not yet exist:")
print(rolling_7.head(8).to_string())
print(f"\nNaN count: {int(rolling_7.isna().sum())}, which is window - 1 = 6.")
print("\nThat is correct, not a defect: a 7-day average of 3 days would be a")
print("different quantity. min_periods= relaxes it if partial windows are")
print("acceptable to you.")

fig, ax = plt.subplots(figsize=(13, 4.2))
ax.plot(series.index, series.to_numpy(), color="#C9CBD3", linewidth=0.8,
        label="daily")
ax.plot(rolling_7_centred.index, rolling_7_centred.to_numpy(),
        color=PALETTE[0], linewidth=1.6, label="7-day, centred")
ax.plot(rolling_30.index, rolling_30.to_numpy(),
        color=PALETTE[3], linewidth=2, label="30-day, centred")

ax.set_title("A rolling mean separates the trend from the cycle", loc="left")
ax.set_xlabel("date")
ax.set_ylabel("visits")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

print("The 7-day window removes the weekly cycle exactly, because the window")
print("is one full cycle. What is left is the trend and the examination peaks.")
print("The 30-day window smooths the peaks away too.")
print("\nWindow length is a choice about WHICH pattern you want to remove.")

In [ ]:
# center=True versus the default, and why it matters.
fig, ax = plt.subplots(figsize=(11, 3.8))

window = series.loc["2026-05":"2026-07"]
trailing = window.rolling(14).mean()
centred = window.rolling(14, center=True).mean()

ax.plot(window.index, window.to_numpy(), color="#C9CBD3", linewidth=0.8,
        label="daily")
ax.plot(trailing.index, trailing.to_numpy(), color=PALETTE[1],
        linewidth=1.8, label="14-day trailing")
ax.plot(centred.index, centred.to_numpy(), color=PALETTE[2],
        linewidth=1.8, label="14-day centred")

ax.set_title("Trailing windows lag; centred windows do not", loc="left")
ax.set_ylabel("visits")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

print("The trailing line peaks about a week after the centred one, because")
print("each value summarises the fortnight ENDING that day.")
print("\nWhich to use is not a style choice:")
print("  DESCRIBING the past      -> center=True, it lines up with events")
print("  FORECASTING, or any model that must not see the future")
print("                           -> leave it off. A centred window at day t")
print("                              contains days after t, which you would")
print("                              not have had at the time.")

## 2.8 Comparing with the past

In [ ]:
frame = visits.copy()
frame["yesterday"] = frame["visits"].shift(1)
frame["change"] = frame["visits"].diff()
frame["pct_change"] = frame["visits"].pct_change() * 100
frame["vs_last_week"] = frame["visits"].diff(7)

print(frame.loc["2026-03-02":"2026-03-10",
                ["visits", "yesterday", "change", "pct_change", "vs_last_week"]]
      .round(1).to_string())

print("\nshift(1) puts yesterday's value on today's row. diff() is the")
print("difference between them, so diff() is series - series.shift(1).")

In [ ]:
# On seasonal data, diff(1) mostly measures the day of the week.
day_one = frame.groupby(frame.index.dayofweek)["change"].mean()
day_seven = frame.groupby(frame.index.dayofweek)["vs_last_week"].mean()

names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
print(f"{'day':<6}{'mean diff(1)':>15}{'mean diff(7)':>15}")
print("-" * 36)
for i, name in enumerate(names):
    print(f"{name:<6}{day_one.get(i, np.nan):>15.1f}{day_seven.get(i, np.nan):>15.1f}")

print("\ndiff(1) swings wildly by day of week: large and negative going into")
print("the weekend, large and positive coming out of it. It is measuring the")
print("weekly cycle, not any change in demand.")
print("\ndiff(7) compares each day with the SAME DAY last week, so the cycle")
print("cancels and what is left is the underlying movement.")
print("\nOn seasonal data, compare like with like.")

---

# Part 3 — Guided workshop (in class)

Four tasks. Each has a verification cell.

### Task 1 — Parse and index

**Problem.** The frame below has dates as text in **day-first** format. Parse them correctly
and build `indexed`: the frame with a sorted `DatetimeIndex`.

Then set `march_rows` to the number of rows falling in March 2026.

**Hints**

- `pd.to_datetime(..., format="%d/%m/%Y")`.
- `.set_index("date").sort_index()`.
- Partial string indexing: `indexed.loc["2026-03"]`.

**Expected output.** A sorted index and a count, followed by `Task 1 complete`.

In [ ]:
messy_dates = pd.DataFrame({
    "date": ["05/03/2026", "01/03/2026", "12/04/2026", "28/02/2026", "31/03/2026"],
    "visits": [410, 385, 502, 330, 455],
})

# TODO 1a: parse the date column, day-first, and build a sorted DatetimeIndex.
indexed = None

# TODO 1b: how many rows fall in March 2026?
march_rows = None

print(indexed)
print(f"\nrows in March 2026: {march_rows}")

In [ ]:
# Verification for Task 1.
want = (messy_dates.assign(date=pd.to_datetime(messy_dates["date"], format="%d/%m/%Y"))
        .set_index("date").sort_index())

problems = []
if indexed is None:
    problems.append("indexed is still None")
elif not isinstance(indexed.index, pd.DatetimeIndex):
    problems.append(f"the index should be a DatetimeIndex, got "
                    f"{type(indexed.index).__name__}")
elif not indexed.index.is_monotonic_increasing:
    problems.append("the index is not sorted — call .sort_index()")
elif not indexed.index.equals(want.index):
    problems.append("the parsed dates do not match. Check the format: these are "
                    "day-first, so 05/03/2026 is 5 March, not 3 May")

if march_rows is None:
    problems.append("march_rows is still None")
elif int(march_rows) != 3:
    problems.append(f"march_rows should be 3, got {march_rows}")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 1 complete.")

### Task 2 — Resample three ways

**Problem.** From `series` (the daily visits), build:

1. `weekly_total` — total visits per week.
2. `monthly_avg` — mean daily visits per month.
3. `n_months` — how many periods `monthly_avg` has.

**Hints**

- `series.resample("W").sum()` and `series.resample("ME").mean()`.
- Remember the rename: month end is `"ME"`, not `"M"`.
- A count is usually summed; a rate is usually averaged.

**Expected output.** Two series and a count, followed by `Task 2 complete`.

In [ ]:
# TODO 2a: total visits per week.
weekly_total = None

# TODO 2b: mean daily visits per month.
monthly_avg = None

# TODO 2c: how many months?
n_months = None

print("Weekly totals, first four:")
print(weekly_total.head(4) if weekly_total is not None else None)
print(f"\nMonths: {n_months}")

In [ ]:
# Verification for Task 2.
want_weekly = series.resample("W").sum()
want_monthly = series.resample("ME").mean()

problems = []
if weekly_total is None:
    problems.append("weekly_total is still None")
elif not np.allclose(weekly_total.to_numpy(), want_weekly.to_numpy()):
    if np.allclose(weekly_total.to_numpy(), series.resample("W").mean().to_numpy()):
        problems.append("weekly_total is the MEAN — a weekly figure for a count "
                        "should be the sum")
    else:
        problems.append("weekly_total does not match the per-week sums")

if monthly_avg is None:
    problems.append("monthly_avg is still None")
elif not np.allclose(monthly_avg.to_numpy(), want_monthly.to_numpy()):
    problems.append("monthly_avg does not match the per-month means")

if n_months is None:
    problems.append("n_months is still None")
elif int(n_months) != 12:
    problems.append(f"n_months should be 12, got {n_months}")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print(f"Task 2 complete. {len(weekly_total)} weeks, {n_months} months.")

### Task 3 — Smooth, and keep the alignment honest

**Problem.** Build two rolling means of `series`:

1. `smooth_centred` — a 7-day **centred** rolling mean.
2. `smooth_trailing` — a 7-day **trailing** rolling mean (the default).

Then set `n_missing_trailing` to the number of `NaN` values in `smooth_trailing`.

**Hints**

- `series.rolling(7, center=True).mean()` and `series.rolling(7).mean()`.
- The leading `NaN` count is `window - 1`.

**Expected output.** Two series and a count, followed by `Task 3 complete`.

In [ ]:
# TODO 3a: 7-day centred rolling mean.
smooth_centred = None

# TODO 3b: 7-day trailing rolling mean.
smooth_trailing = None

# TODO 3c: how many NaN values does the trailing version have?
n_missing_trailing = None

print(f"NaN in trailing : {n_missing_trailing}")
print(f"NaN in centred  : "
      f"{int(smooth_centred.isna().sum()) if smooth_centred is not None else None}")

In [ ]:
# Verification for Task 3.
want_centred = series.rolling(7, center=True).mean()
want_trailing = series.rolling(7).mean()

problems = []
if smooth_centred is None:
    problems.append("smooth_centred is still None")
elif not np.allclose(smooth_centred.to_numpy(), want_centred.to_numpy(),
                     equal_nan=True):
    problems.append("smooth_centred does not match a 7-day centred window")

if smooth_trailing is None:
    problems.append("smooth_trailing is still None")
elif not np.allclose(smooth_trailing.to_numpy(), want_trailing.to_numpy(),
                     equal_nan=True):
    problems.append("smooth_trailing does not match a 7-day trailing window")

if n_missing_trailing is None:
    problems.append("n_missing_trailing is still None")
elif int(n_missing_trailing) != 6:
    problems.append(f"n_missing_trailing should be 6 (window - 1), "
                    f"got {n_missing_trailing}")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 3 complete.")
    print("\nBoth have 6 missing values, but in different places: the trailing")
    print("version loses the first 6 days, the centred version loses 3 at each end.")

### Task 4 — Compare like with like

**Problem.** The library wants to know whether demand is genuinely rising, separately from the
weekly cycle.

1. `change_1` — the day-on-day change in `series`.
2. `change_7` — the change against the same day one week earlier.
3. `sd_ratio` — the standard deviation of `change_1` divided by that of `change_7`, as a float.

**Hints**

- `series.diff()` and `series.diff(7)`.
- `.std()` on each, then divide.
- If `diff(1)` is mostly measuring the weekly cycle, its standard deviation will be much the
  larger of the two.

**Expected output.** A ratio above 1, followed by `Task 4 complete`.

In [ ]:
# TODO 4a: day-on-day change.
change_1 = None

# TODO 4b: change against the same day last week.
change_7 = None

# TODO 4c: ratio of the two standard deviations.
sd_ratio = None

print(f"sd of diff(1) : "
      f"{change_1.std():.1f}" if change_1 is not None else "sd of diff(1) : None")
print(f"sd of diff(7) : "
      f"{change_7.std():.1f}" if change_7 is not None else "sd of diff(7) : None")
print(f"ratio         : {sd_ratio}")

In [ ]:
# Verification for Task 4.
want_1 = series.diff()
want_7 = series.diff(7)
want_ratio = float(want_1.std() / want_7.std())

problems = []
if change_1 is None or not np.allclose(change_1.to_numpy(), want_1.to_numpy(),
                                       equal_nan=True):
    problems.append("change_1 should be series.diff()")
if change_7 is None or not np.allclose(change_7.to_numpy(), want_7.to_numpy(),
                                       equal_nan=True):
    problems.append("change_7 should be series.diff(7) — the same day last week")
if sd_ratio is None:
    problems.append("sd_ratio is still None")
elif not np.isclose(float(sd_ratio), want_ratio):
    problems.append(f"sd_ratio should be about {want_ratio:.3f}")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print(f"Task 4 complete. Ratio = {want_ratio:.2f}")
    print("\nThe day-on-day change is far more variable, because most of what")
    print("it measures is the weekly cycle rather than any change in demand.")
    print("diff(7) compares like with like and is the honest measure here.")

---

# Part 4 — Homework

Run **Restart Kernel and Run All Cells** before submitting, then push to your
`puc-data-analysis` repository.

## Problem 1 — Time features from a date column

**Problem.** Write `add_time_features(df, date_col)`.

It returns a **new** frame with the original columns plus extracted time features, ready for
grouping and plotting.

**Requirements.** Add exactly these columns, in this order, after the existing ones:

| Column | Type | Value |
|---|---|---|
| `year` | integer | Calendar year |
| `month` | integer | Month number, 1–12 |
| `day_name` | string | `"Monday"` … `"Sunday"` |
| `dayofweek` | integer | Monday = 0 … Sunday = 6 |
| `quarter` | integer | 1–4 |
| `is_weekend` | `bool` | `True` for Saturday and Sunday |

**Requirements on behaviour**

- If `date_col` is not already a datetime, convert it with `errors="coerce"`.
- Rows whose date is `NaT` keep `NaT`/missing values in the new columns rather than raising.
- Raise `KeyError` if `date_col` is absent.
- Do not modify the input frame, and do not set an index.

**Hints**

- `pd.api.types.is_datetime64_any_dtype(series)` tests whether conversion is needed.
- The `.dt` accessor gives every one of these directly.
- `dayofweek >= 5` gives the weekend flag. With `NaT` present the comparison yields `False`,
  which is acceptable — but the numeric columns will be float because `NaN` forces promotion,
  and that is expected.
- Work on `df.copy()`.

In [ ]:
# Problem 1 — starter code.

def add_time_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """Return a copy of df with time features extracted from date_col.

    Parameters
    ----------
    df : pandas.DataFrame
        The data. Not modified.
    date_col : str
        Name of the date column.

    Returns
    -------
    pandas.DataFrame
        A new frame with six extra columns.

    Raises
    ------
    KeyError
        If date_col is absent.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete add_time_features()")


featured = add_time_features(raw, "date")
print(featured.head(8).to_string(index=False))

print("\nMean visits by day of week:")
print(featured.groupby("day_name")["visits"].mean().round(1).to_string())

In [ ]:
# Problem 1 — validation. All assertions must pass.

sample = pd.DataFrame({
    "when": pd.to_datetime(["2026-01-05",   # a Monday
                            "2026-03-14",   # a Saturday
                            "2026-12-31"]), # a Thursday
    "value": [1, 2, 3],
})

out = add_time_features(sample, "when")

# --- structure ---
assert isinstance(out, pd.DataFrame)
expected_new = ["year", "month", "day_name", "dayofweek", "quarter", "is_weekend"]
assert list(out.columns) == ["when", "value"] + expected_new, (
    f"Original columns first, then the six new ones in order; got {list(out.columns)}"
)
assert len(out) == len(sample), "No rows may be added or dropped"
assert not isinstance(out.index, pd.DatetimeIndex), "Do not set an index"

# --- values ---
assert out["year"].tolist() == [2026, 2026, 2026]
assert out["month"].tolist() == [1, 3, 12]
assert out["quarter"].tolist() == [1, 1, 4]
assert out["day_name"].tolist() == ["Monday", "Saturday", "Thursday"]
assert out["dayofweek"].tolist() == [0, 5, 3], "Monday is 0, Sunday is 6"
assert out["is_weekend"].tolist() == [False, True, False]
assert out["is_weekend"].dtype == bool, "is_weekend must be a boolean column"

# --- the input is untouched ---
assert list(sample.columns) == ["when", "value"], (
    "add_time_features must not modify its input"
)

# --- a text date column is converted ---
as_text = pd.DataFrame({"when": ["2026-01-05", "2026-03-14"], "value": [1, 2]})
converted = add_time_features(as_text, "when")
assert converted["day_name"].tolist() == ["Monday", "Saturday"], (
    "A text date column must be converted before the features are extracted"
)

# --- an unparseable date becomes NaT and does not raise ---
with_bad = pd.DataFrame({"when": ["2026-01-05", "not a date"], "value": [1, 2]})
handled = add_time_features(with_bad, "when")
assert len(handled) == 2, "The bad row must be kept, not dropped"
assert pd.isna(handled.loc[1, "day_name"]) or handled.loc[1, "day_name"] is None, (
    "A NaT date should give a missing day_name, not a crash"
)
assert pd.isna(handled.loc[1, "year"]), "A NaT date should give a missing year"
assert not pd.isna(handled.loc[0, "year"]), "The good row must still be populated"

# --- a missing column raises ---
try:
    add_time_features(sample, "nope")
except KeyError:
    pass
else:
    raise AssertionError("A missing date_col must raise KeyError")

# --- it works on the notebook's own data ---
real = add_time_features(raw, "date")
assert len(real) == len(raw)
assert real["is_weekend"].sum() == raw["is_weekend"].sum(), (
    "The computed weekend flag should match the one in the source data"
)

print("Problem 1: all assertions passed.")

---

## Problem 2 — A resampling report

Section 2.5 argued that choosing a frequency is choosing what to discard. This function makes
that visible for any series.

**Problem.** Write `resample_report(series, freqs, how="mean")`.

**Requirements.** Return a `DataFrame` with **one row per frequency**, indexed by the
frequency alias with the index named `"freq"`, and exactly these columns:

| Column | Type | Value |
|---|---|---|
| `n_periods` | `int` | Number of periods after resampling |
| `mean` | `float` | Mean of the resampled series |
| `std` | `float` | Standard deviation of the resampled series |
| `min` | `float` | Minimum |
| `max` | `float` | Maximum |
| `pct_variation_kept` | `float` | The resampled `std` as a percentage of the **original** series' `std`, rounded to 1 dp |

Rows must appear in the order given in `freqs`.

**Requirements on behaviour**

- `how` may be `"mean"`, `"sum"`, `"min"` or `"max"`; anything else raises `ValueError`.
- Raise `TypeError` if the series does not have a `DatetimeIndex`.
- Raise `ValueError` if `freqs` is empty.
- An invalid frequency alias must propagate pandas' own `ValueError` — **do not catch it.**
  A silent skip would hide the `M` versus `ME` problem from §2.4, which is exactly the error
  this function should make loud.

Do not modify the input.

**Hints**

- `getattr(series.resample(freq), how)()` applies the named aggregation.
- Compute the original standard deviation **once**, before the loop.
- `pct_variation_kept` is what §2.5 was measuring. Note that it does **not** always fall: a
  series whose variation is *all* trend keeps nearly 100% at every frequency, because a trend
  is visible at any resolution. It collapses only when there is high-frequency structure to
  destroy. The tests check both cases.

In [ ]:
# Problem 2 — starter code.

def resample_report(series: pd.Series, freqs, how: str = "mean") -> pd.DataFrame:
    """Summarise a series at several frequencies, and how much variation survives.

    Parameters
    ----------
    series : pandas.Series
        A series with a DatetimeIndex. Not modified.
    freqs : list of str
        Frequency aliases, e.g. ["D", "W", "ME"].
    how : str
        Aggregation: "mean", "sum", "min" or "max".

    Returns
    -------
    pandas.DataFrame
        One row per frequency, indexed by the alias.

    Raises
    ------
    TypeError
        If the series does not have a DatetimeIndex.
    ValueError
        If freqs is empty, or how is not recognised.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete resample_report()")


report = resample_report(series, ["D", "W", "ME", "QE"])
print(report.to_string())
print("\npct_variation_kept is the story of section 2.5, as a number.")

In [ ]:
# Problem 2 — validation. All assertions must pass.

# A series dominated by a WEEKLY cycle, with only a gentle trend. Resampling
# to weekly or monthly should therefore destroy most of its variation.
_index = pd.date_range("2026-01-01", periods=364, freq="D")
_weekend = _index.dayofweek.to_numpy() >= 5
test_series = pd.Series(
    100 + np.arange(364) * 0.02 + np.where(_weekend, -50.0, 20.0),
    index=_index,
)

out = resample_report(test_series, ["D", "W", "ME"])

# --- structure ---
assert isinstance(out, pd.DataFrame), "resample_report must return a DataFrame"
assert list(out.index) == ["D", "W", "ME"], (
    f"One row per frequency, in the order given; got {list(out.index)}"
)
assert out.index.name == "freq", f"The index must be named 'freq'; got {out.index.name!r}"

expected_cols = ["n_periods", "mean", "std", "min", "max", "pct_variation_kept"]
assert list(out.columns) == expected_cols, (
    f"Columns must be exactly {expected_cols}; got {list(out.columns)}"
)

# --- values ---
assert out.loc["D", "n_periods"] == 364, "Daily resampling of daily data changes nothing"
assert out.loc["ME", "n_periods"] == 12, "A year has 12 month-ends"
assert isinstance(out.loc["D", "n_periods"], (int, np.integer))

assert np.isclose(out.loc["D", "mean"], test_series.mean())
assert np.isclose(out.loc["D", "min"], test_series.min())
assert np.isclose(out.loc["D", "max"], test_series.max())

# --- variation kept collapses as the frequency coarsens ---
assert np.isclose(out.loc["D", "pct_variation_kept"], 100.0), (
    "Resampling daily data to daily keeps all the variation"
)
assert out.loc["W", "pct_variation_kept"] < 30.0, (
    f"This series is dominated by a weekly cycle, so weekly resampling should "
    f"destroy most of its variation. Got {out.loc['W', 'pct_variation_kept']}%"
)
assert out.loc["ME", "pct_variation_kept"] < out.loc["W", "pct_variation_kept"], (
    "Coarser frequencies keep less variation — this is the point of §2.5"
)
assert round(out.loc["ME", "pct_variation_kept"], 1) == out.loc["ME", "pct_variation_kept"], (
    "pct_variation_kept must be rounded to 1 decimal place"
)

# --- the contrast: a pure trend loses NOTHING when resampled ---
# All of its variation IS the trend, and a trend survives at every frequency.
# This is the other half of §2.5, and it is why "resampling loses information"
# is a statement about WHICH information, not a general law.
ramp = pd.Series(np.arange(364, dtype=float), index=_index)
ramp_report = resample_report(ramp, ["D", "ME"])
assert ramp_report.loc["ME", "pct_variation_kept"] > 90.0, (
    f"A pure linear trend keeps essentially all its variation under resampling, "
    f"because the trend is visible at every frequency. Got "
    f"{ramp_report.loc['ME', 'pct_variation_kept']}%. If this fails, "
    f"pct_variation_kept is probably being computed against the wrong baseline."
)

# --- how= is honoured ---
summed = resample_report(test_series, ["ME"], how="sum")
assert summed.loc["ME", "mean"] > out.loc["ME", "mean"], (
    "Summing a month of values gives a much larger figure than averaging them"
)
maxed = resample_report(test_series, ["ME"], how="max")
assert np.isclose(maxed.loc["ME", "max"], test_series.max()), (
    "Taking the max of each month and then the max of those gives the overall max"
)

# --- the frequency order is preserved, not sorted ---
reordered = resample_report(test_series, ["ME", "D", "W"])
assert list(reordered.index) == ["ME", "D", "W"], (
    "Rows must follow the order of `freqs`, not be sorted"
)

# --- errors ---
try:
    resample_report(test_series, ["D"], how="median")
except ValueError:
    pass
else:
    raise AssertionError("An unrecognised `how` must raise ValueError")

try:
    resample_report(test_series, [])
except ValueError:
    pass
else:
    raise AssertionError("An empty freqs list must raise ValueError")

try:
    resample_report(pd.Series([1, 2, 3]), ["D"])
except TypeError:
    pass
else:
    raise AssertionError("A series without a DatetimeIndex must raise TypeError")

# --- an invalid alias must PROPAGATE, not be swallowed ---
try:
    resample_report(test_series, ["M"])
except ValueError as exc:
    assert "M" in str(exc) or "frequency" in str(exc).lower(), (
        "The ValueError from pandas should reach the caller unchanged, so that "
        "'Invalid frequency: M' is visible — do not catch and re-raise with a "
        "different message"
    )
else:
    raise AssertionError(
        "'M' was removed in pandas 3.0 and must raise. Do not catch it: a silent "
        "skip would hide exactly the problem section 2.4 warned about."
    )

# --- the input is untouched ---
before = test_series.copy()
_ = resample_report(test_series, ["W"])
assert test_series.equals(before), "resample_report must not modify its input"

print("Problem 2: all assertions passed.")

---

## Written question

Answer in the Markdown cell below, in 120–200 words.

Section 2.5 showed that the weekly cycle — the largest single pattern in the library data —
is completely absent from the monthly series.

1. Explain why it vanishes **exactly**, rather than merely being reduced.
2. A colleague reports "library use is steady through the year, with a slow rise" based on the
   quarterly chart. What is wrong with the claim, and what would you ask to see?
3. You are given a series of hourly electricity readings for one building over two years, and
   asked to find out when the building is busiest. Which frequency would you look at first,
   and why?

*Replace this text with your answer.*

**1.**

**2.**

**3.**

---

## Submission checklist

- [ ] All four workshop tasks print their `complete` message.
- [ ] Problem 1 prints `all assertions passed`.
- [ ] Problem 2 prints `all assertions passed`.
- [ ] The written question is answered in 120–200 words.
- [ ] **Restart Kernel and Run All Cells** completes with no errors.
- [ ] Committed and pushed to your `puc-data-analysis` repository.

## Looking ahead to Week 11

Week 11 is Plotly: interactive charts that a reader can hover over, zoom into and filter. The
time series from today is the natural example — zooming into a year of daily data is exactly
the problem that resampling solved by throwing information away, and interactivity solves by
keeping it.

Before the session, read McKinney §9.3.

---

## References

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. §11.1–11.7.

Pfaffelhuber, P. *Python for Data Analysis* (course notes). §7.10.
Available at `https://pfaffelh.github.io/python_for_data/`.